In [0]:
import os
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql import Window
import pyspark.sql.utils;
from pyspark.sql.types import StructType, StringType, IntegerType, TimestampType
from pyspark.sql.functions import concat, lit, col, upper, max, when, row_number, round, length, expr, regexp_extract, explode, to_date, first, udf, current_timestamp, round, regexp_replace, format_string, date_format, trim
from datetime import datetime, timedelta
import numpy as np
from pyspark.sql import SparkSession
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

In [0]:
try:
    verbose_mode = dbutils.widgets.get("verbose_mode");
except:
    verbose_mode = 'debug'
try:
    current_division = dbutils.widgets.get("division");
except:
    current_division = 'mal'
try:
    current_environment = dbutils.widgets.get("environment");
except:
    current_environment = 'dev'
try:
    current_project = dbutils.widgets.get("project");
except:
    current_project = 'maite_bi'
try:
    current_production_line = dbutils.widgets.get("production_line");
except:
    current_production_line = 'self_service'

current_catalog = current_division + '_' + current_project + '_' + current_environment;

current_schema = current_production_line;

current_location = 'abfss://' + current_project + '@adlsdpcom'+ current_environment +f'data.dfs.core.windows.net/' + current_production_line + '/'

if verbose_mode == 'debug':
    display("Debug Mode")
    display(f"current_division : {current_division}")
    display(f"current_environment : {current_environment}")
    display(f"current_project : {current_project}")
    display(f"current_production_line : {current_production_line}")
    display(f"current_catalog : {current_catalog}")
    display(f"current_schema : {current_schema}")
    display(f"current_location : {current_location}")

In [0]:
def delta_table_sql_update_table(source_table, target_table, business_keys, excluded_fields, update_option, verbose_mode):
    """
    Met à jour la table cible avec les données de la table source en excluant les champs exclus et les champs absents de la cible.

    Args:
        source_table: référence de la delta table ou de la vue temporaire source
        target_table: référence de la delta cible
        business_keys: description de la clé de jointure sous la forme [('batch_id',)], ('...',), ('...',)]
        excluded_fields: description des champs à exclure, a minima, les champs d'horodatage [('created_at',), ('updated_at',), ('deleted_at',), ('max_date_time',), ...]
        update_option : mode de mise à jour souhaité ("update" : n'effectue que les mises à jour, "upsert" : effectue les mises à jour et insert, "merge" : effectue les mises à jour et insert et les delete)

    Returns:
    -1 : KO
    0  : No Data from source 
    1 : OK
    """
    function_result = -1
    schema = StructType().add("field_name", StringType())
    # Création des dataframes
    source_df = spark.sql(f"""SELECT * FROM {source_table} LIMIT 1""")
    if source_df.isEmpty():
        function_result = 0
        print("La source est vide")
    else:
        business_keys_df = spark.createDataFrame(business_keys, schema)
        excluded_fields_df = spark.createDataFrame(excluded_fields, schema)

        target_df = spark.sql(f"""SELECT * FROM {target_table} LIMIT 1""")
        target_schema = target_df.schema
        # Champs de la table cible
        target_fields_df = spark.createDataFrame([(c,) for c in target_df.columns], ["field_name"])
        # Typage de la source
        for column_name in source_df.columns:
            if column_name in target_fields_df.columns:
                source_df = source_df.withColumn(column_name, source_df[column_name].cast(target_schema[column_name].dataType)) 
        # Champs de la table source
        source_fields_df = spark.createDataFrame([(c,) for c in source_df.columns], ["field_name"])
        # Champs de la table cible
        target_fields_df = spark.createDataFrame([(c,) for c in target_df.columns], ["field_name"])
        # On supprime les champs absents dans la cible
        common_fields_df = source_fields_df.join(target_fields_df, "field_name", "left_semi")
        # On supprime les champs exclus
        all_fields_df = common_fields_df.join(excluded_fields_df, 'field_name', "left_anti")
        # Champs retenus hors clés
        field_list_df = all_fields_df.join(business_keys_df, 'field_name', "left_anti")
        # Création de la clause SELECT        
        select_clause_df = all_fields_df.withColumn('field_name', col('field_name'))
        select_clause_lst = [row.field_name for row in select_clause_df.collect()]
        select_clause_str = ", \n".join(select_clause_lst)
        # Création de la clause DE JOINTURE
        join_clause_df = business_keys_df.withColumn('field_name', concat(lit("target."), col('field_name'), lit(" = source."), col('field_name')))
        join_clause_lst = [row.field_name for row in join_clause_df.collect()]
        join_clause_str = "\n AND ".join(join_clause_lst)
        # Création de la liste des champs pour l'insertion
        insert_left_clause_df = all_fields_df.withColumn('field_name', col('field_name'))
        insert_left_clause_lst = [row.field_name for row in insert_left_clause_df.collect()]
        insert_left_clause_str = ", \n".join(insert_left_clause_lst)
        insert_left_clause_str += "\n, created_at"
        # Création de la liste des champs à insérer
        insert_right_clause_df = all_fields_df.withColumn('field_name', concat(lit("source."), col('field_name')))
        insert_right_clause_lst = [row.field_name for row in insert_right_clause_df.collect()]
        insert_right_clause_str = ", \n".join(insert_right_clause_lst)
        insert_right_clause_str += "\n, current_timestamp()"
        # Création de la condition pour l'insertion
        update_condition_clause_df = field_list_df.withColumn('field_name', concat(lit("(target."), col('field_name'), lit("<> source."), col('field_name'), lit(") or (target."), col('field_name'), lit(" IS NULL AND source."), col('field_name'), lit(" IS NOT NULL) OR (target."), col('field_name'), lit(" IS NOT NULL AND source."), col('field_name'), lit(" IS NULL)")))
        update_condition_clause_lst = [row.field_name for row in update_condition_clause_df.collect()]
        update_condition_clause_str = "\n OR ".join(update_condition_clause_lst)    #Problème ici
        # Création de la clause d'UPDATE
        update_clause_df = field_list_df.withColumn('field_name', concat(lit("target."), col('field_name'), lit(" = source."), col('field_name')))
        update_clause_lst = [row.field_name for row in update_clause_df.collect()]
        update_clause_str = ", \n ".join(update_clause_lst)
        update_clause_str += "\n, target.updated_at = current_timestamp()"
        #Définition de la requête de MERGE
        merge_query_sql = f"""MERGE INTO 
            {target_table} AS target
        USING
            {source_table} AS source
        ON {join_clause_str}
        WHEN MATCHED AND ({update_condition_clause_str}) 
            THEN UPDATE SET {update_clause_str}"""
        
        if update_option =="upsert" or update_option =="merge" :
            merge_query_sql = merge_query_sql + f"""
        WHEN NOT MATCHED BY TARGET 
            THEN INSERT ({insert_left_clause_str}) 
                VALUES ({insert_right_clause_str})"""
        
        if update_option =="merge" :
            merge_query_sql = merge_query_sql + f"""
        WHEN NOT MATCHED BY SOURCE AND deleted = FALSE
            THEN UPDATE SET
                deleted=TRUE,
                deleted_at = current_timestamp()
            """

        merge_query_sql = merge_query_sql + """
    ;"""

        if verbose_mode == 'debug':
            print('Requête de merge')
            display(merge_query_sql)  

        try:
            spark.sql(merge_query_sql)
            function_result = 1
        except Exception as e:
            print("erreur de merge:", e)
            function_result = -1
    return function_result

Source(s)

In [0]:
if current_environment =='preprd':
    source = f"""ext_mal_psql_maite_vision_board_test.public"""    
else:
    source = f"""ext_mal_psql_maite_vision_board_{current_environment}.public"""

In [0]:
source_gold = f"""mal_maite_common_{current_environment}.gold"""

In [0]:
source_mal_maite = f"""mal_maite_{current_environment}"""

batch_report

In [0]:
current_process="fact_batch_report"

In [0]:
spark.sql(f"TRUNCATE TABLE {current_catalog}.{current_schema}.{current_process}");

In [0]:
batches = f"""
SELECT
  b.id_batch,
  b.batch_number AS batch,
  b.mes_number,
  b.fabrication_order_number,
  ppl.name AS site_de_production,
  ppt.code AS type_de_production,
  b.harvest AS annee_de_recolte,
  b.planned_date AS debut_de_production,
  rs.name AS cahier_des_charges,
  gs.code AS espece,
  gv.code AS variete

FROM {source}.batches b

LEFT JOIN {source}.plants_production_lines ppl ON ppl.id_plant_production_line = b.production_line
LEFT JOIN {source}.parameters_production_types ppt ON ppt.id_parameter_production_type = b.production_type
LEFT JOIN {source}.requirement_specifications rs ON rs.id_requirement_specification = b.requirement_specifications AND rs.deleted = false
LEFT JOIN {source}.goods_varieties gv ON gv.id_good_variety = b.variety
LEFT JOIN {source}.goods_species gs ON gs.id_good_specy = gv.specy

WHERE b.deleted = false
"""

df_batches = spark.sql(batches)
df_batches.createOrReplaceTempView("batches")


# Méthode de définition des dates de fin de production
date_fin_production = f"""
SELECT b.id_batch, b.planned_date, 
        l.kiln_unload_end_date, 
        l.kiln_unload_start_date, 
        l.germ_unload_start_date, 
        l.steep_c1_o1_filling_start_date
FROM {source}.batches b
LEFT JOIN {source_gold}.localizations_pivot l ON b.id_batch = l.batch_id
"""

df_date_fin_production = spark.sql(date_fin_production)

# Calcul du champ fin_de_production selon la priorité
df_date_fin_production = df_date_fin_production.withColumn(
    "fin_de_production",
    when(col("kiln_unload_end_date").isNotNull(), col("kiln_unload_end_date"))
    .when(col("kiln_unload_start_date").isNotNull(), col("kiln_unload_start_date"))
    .when(col("germ_unload_start_date").isNotNull(), col("germ_unload_start_date") + expr("INTERVAL 2 DAYS"))
    .when(col("steep_c1_o1_filling_start_date").isNotNull(), col("steep_c1_o1_filling_start_date") + expr("INTERVAL 8 DAYS"))
    .otherwise(col("planned_date") + expr("INTERVAL 8 DAYS")))

# Convertir la colonne 'calculation_date' en date uniquement (sans les heures)
df_date_fin_production = df_date_fin_production.withColumn('fin_de_production', col('fin_de_production').cast('date'))

df_date_fin_production.createOrReplaceTempView("date_fin_production")


# Jointure pour ajouter les dates de fin de production
main = f"""
SELECT b.*, dfp.fin_de_production
FROM batches b
LEFT JOIN date_fin_production dfp ON b.id_batch = dfp.id_batch
"""
df_main = spark.sql(main)

# Conversion de toutes les colonnes en STRING
df_main = df_main.select([col(c).cast("STRING").alias(c) for c in df_main.columns])

# Liste des colonnes à dépivoter (toutes sauf "id_batch" et "batch")
columns_to_unpivot = [col for col in df_main.columns if col not in ["id_batch", "batch"]]

stack_expr = ", ".join([f"'{col}', {col}" for col in columns_to_unpivot])

df_main = df_main.selectExpr("id_batch", "batch", f"stack({len(columns_to_unpivot)}, {stack_expr}) as (parameter, value)").dropna(subset=["value"])

# Renomme les valeurs
df_main = df_main.withColumn(
    "parameter",
    when(col("parameter") == "fin_de_production", "Fin de production")
    .when(col("parameter") == "mes_number", "Lot mes")
    .when(col("parameter") == "debut_de_production", "Début de production")
    .when(col("parameter") == "fabrication_order_number", "Ordre de fabrication")
    .when(col("parameter") == "site_de_production", "Site de production")
    .when(col("parameter") == "type_de_production", "Type de production")
    .when(col("parameter") == "annee_de_recolte", "Année de récolte")
    .when(col("parameter") == "cahier_des_charges", "Cahier des charges")
    .when(col("parameter") == "espece", "Espèce")
    .when(col("parameter") == "variete", "Variété")
    .otherwise(col("parameter"))
)

In [0]:
parameter_goods_malt = f"""
SELECT b.id_batch, b.batch_number AS batch, pvt.label AS parameter, me.value
FROM {source}.batches b

JOIN {source}.manual_entries me ON me.batch = b.id_batch AND me.deleted = false

JOIN {source}.parameters_variables pv ON pv.id_parameter_variable = me.parameter AND pv.deleted = false

JOIN {source}.parameters_variables_translations pvt ON pvt.parameter_variable = pv.id_parameter_variable AND pvt.language = 1 AND pvt.deleted = false

WHERE b.deleted = false AND pv.code IN ('malt_yield_r2', 'goods_weight', 'goods_moisture', 'malt_weight')
"""

df_parameter_goods_malt = spark.sql(parameter_goods_malt)


# Ajout malt_moisture via table gold
malt_moisture_gold = f"""
SELECT b.id_batch, b.batch_number AS batch, mqr.malt_moisture
FROM {source_gold}.malt_quality_results mqr
JOIN {source}.batches b ON b.fabrication_order_number = mqr.fabrication_order AND b.deleted = false
"""
df_malt_moisture_gold = spark.sql(malt_moisture_gold)

df_malt_moisture_gold = df_malt_moisture_gold.withColumn("parameter", lit("Moisture Malt")) \
                                  .withColumnRenamed("malt_moisture", "value") \
                                  .select("id_batch", "batch", "parameter", "value")

# Union des deux dataframes
df_parameter_goods_malt = df_parameter_goods_malt.unionByName(df_malt_moisture_gold)

df_parameter_goods_malt = df_parameter_goods_malt.withColumn("value", round("value", 2))

# Conversion de toutes les colonnes en STRING
df_parameter_goods_malt = df_parameter_goods_malt.select([col(c).cast("STRING").alias(c) for c in df_parameter_goods_malt.columns])

In [0]:
r2_sec = f"""
SELECT b.id_batch, b.batch_number AS batch, mqr.malt_moisture, pv.code AS parameter, me.value
FROM {source}.batches b
JOIN {source}.manual_entries me ON me.batch = b.id_batch AND me.deleted = false
JOIN {source}.parameters_variables pv ON pv.id_parameter_variable = me.parameter AND pv.deleted = false
LEFT JOIN {source_gold}.malt_quality_results mqr ON mqr.fabrication_order = b.fabrication_order_number
WHERE b.deleted = false AND pv.code IN ('malt_yield_r2', 'goods_weight', 'goods_moisture', 'malt_weight')
"""

df_r2_sec = spark.sql(r2_sec)

# Pivot
df_r2_sec = df_r2_sec.groupBy("id_batch", "batch", "malt_moisture").pivot("parameter").agg(F.first("value"))

# Calcul du r2_sec
df_r2_sec = df_r2_sec.withColumn("R2 Dry",round(
        (col("malt_weight") - (col("malt_weight") * (col("malt_moisture") / 100))) /
        (col("goods_weight") - (col("goods_weight") * (col("goods_moisture") / 100))) * 100, 1))

# Garde seulement les colonnes 'id_batch', 'batch' et 'R2 Dry'
df_r2_sec = df_r2_sec.select("id_batch", "batch", 'R2 Dry')

# Dépivoter la colonne 'R2 Dry'
df_r2_sec = df_r2_sec.selectExpr("id_batch", "batch", "'R2 Dry' as parameter", "`R2 Dry` as value").dropna(subset=["value"])

# Conversion de toutes les colonnes en STRING
df_r2_sec = df_r2_sec.select([col(c).cast("STRING").alias(c) for c in df_r2_sec.columns])

In [0]:
parameter_categ_1 = f"""
SELECT b.id_batch, b.batch_number AS batch, pvt.label AS parameter, me.value

FROM {source}.batches b

JOIN {source}.manual_entries me ON me.batch = b.id_batch
    AND me.deleted = false

JOIN {source}.parameters_variables pv ON pv.id_parameter_variable = me.parameter
    AND pv.category = 1 AND pv.deleted = false

JOIN {source}.parameters_variables_translations pvt ON pvt.parameter_variable = pv.id_parameter_variable
    AND pvt.language = 1 AND pvt.deleted = false

WHERE b.deleted = false AND pv.code NOT IN ('goods_broken_grains_percent', 'goods_bad_grains_percent')
"""

df_parameter_categ_1 = spark.sql(parameter_categ_1)

df_parameter_categ_1 = df_parameter_categ_1.withColumn("value", round("value", 2))

# Supprime les erreurs
df_parameter_categ_1 = df_parameter_categ_1.filter(~((col("parameter") == "Date de calibrage") & (col("value") < 20230101)))

# Conversion en date pour les valeurs de 'Date calibrage'
df_parameter_categ_1 = df_parameter_categ_1.withColumn(
    "value",
    when(
        col("parameter") == "Date de calibrage",
        date_format(
            to_date(format_string("%08d", col("value").cast("long")), "yyyyMMdd"),
            "yyyy-MM-dd"
        )
    ).otherwise(col("value").cast("string"))
)

In [0]:
manual_entries = f"""
SELECT b.id_batch, b.batch_number AS batch, pvt.label AS parameter, me.value
FROM {source}.batches b

JOIN {source}.manual_entries me ON me.batch = b.id_batch AND me.deleted = false

JOIN {source}.parameters_variables pv ON pv.id_parameter_variable = me.parameter
    AND pv.category = 2

JOIN {source}.parameters_variables_translations pvt ON pvt.parameter_variable = pv.id_parameter_variable
    AND pvt.language = 1 AND pvt.deleted = false

WHERE b.deleted = false
"""

df_manual_entries = spark.sql(manual_entries)

df_manual_entries = df_manual_entries.withColumn("value", round("value", 2))

# Conversion de toutes les colonnes en STRING
df_manual_entries = df_manual_entries.select([col(c).cast("STRING").alias(c) for c in df_manual_entries.columns])

In [0]:
quality_results = f"""
SELECT b.id_batch, b.batch_number AS batch, gqr.goods_bad_grains_percent, gqr.goods_broken_grains_percent, mqr.malt_color_ebc, mqr.malt_free_amino_nitrogen, mqr.malt_friability, mqr.malt_moisture AS malt_moisture_quality, mqr.malt_soluble_beta_glucan, mqr.malt_viscosity
FROM {source}.batches b
LEFT JOIN {source_gold}.goods_quality_results gqr ON gqr.fabrication_order = b.fabrication_order_number
LEFT JOIN {source_gold}.malt_quality_results mqr ON mqr.fabrication_order = b.fabrication_order_number
WHERE b.deleted = false
"""

df_quality_results = spark.sql(quality_results)

# Liste des colonnes à dépivoter (toutes sauf "id_batch" et "batch")
columns_to_unpivot = [col for col in df_quality_results.columns if col not in ["id_batch", "batch"]]

stack_expr = ", ".join([f"'{col}', {col}" for col in columns_to_unpivot])

df_quality_results = df_quality_results.selectExpr("id_batch", "batch", f"stack({len(columns_to_unpivot)}, {stack_expr}) as (parameter, value)").dropna(subset=["value"])

# Renomme les valeurs
df_quality_results = df_quality_results.withColumn(
    "parameter",
    when(col("parameter") == "malt_moisture_quality", "Moisture Malt Quality")
    .when(col("parameter") == "malt_soluble_beta_glucan", "malt_soluble_betaglucan")
    .otherwise(col("parameter"))
)

df_quality_results = df_quality_results.withColumn("value", round("value", 2))

df_quality_results.createOrReplaceTempView("quality_results")

quality = f"""
SELECT qr.*, pet.label, pvt.label AS label_2
FROM quality_results qr
LEFT JOIN {source}.parameters_variables pv ON pv.code = qr.parameter AND pv.deleted = false
LEFT JOIN {source}.parameters_variables_translations pvt ON pvt.parameter_variable = pv.id_parameter_variable AND pvt.language = 1 AND pvt.deleted = false
LEFT JOIN {source}.parameters_evaluations pe ON pe.code = qr.parameter AND pe.deleted = false
LEFT JOIN {source}.parameters_evaluations_translations pet ON pet.id_parameter_evaluation = pe.id_parameter_evaluation AND pet.language = 1 AND pet.deleted = false
"""

df_quality = spark.sql(quality)

df_quality = df_quality.withColumn(
    "parameter",
    when(col("label_2").isNotNull(), col("label_2"))
    .when(col("label").isNotNull(), col("label"))
    .otherwise(col("parameter"))).drop("label", "label_2")

# Conversion de toutes les colonnes en STRING
df_quality = df_quality.select([col(c).cast("STRING").alias(c) for c in df_quality.columns])

In [0]:
bqt = f"""
SELECT b.fabrication_order_number, pe.code, btq.upper_limit, btq.lower_limit
FROM {source}.batches_quality_targets btq
JOIN {source}.batches b ON b.id_batch = btq.batch AND b.deleted = false
JOIN {source}.parameters_evaluations pe ON btq.evaluation_parameter = pe.id_parameter_evaluation
WHERE btq.deleted = false AND b.fabrication_order_number IS NOT NULL
"""

df_bqt = spark.sql(bqt)

# Max upper_limit
df_bqt_max = df_bqt.groupBy("fabrication_order_number").pivot("code").agg(F.max("upper_limit"))

# Renommer les colonnes dans df_bqt_max
df_bqt_max = df_bqt_max.withColumnRenamed('malt_color_ebc', 'coloration_EBC_max') \
                       .withColumnRenamed('malt_free_amino_nitrogen', 'FAN_max') \
                       .withColumnRenamed('malt_friability', 'friabilite_max') \
                       .withColumnRenamed('malt_soluble_betaglucan', 'betaG_max') \
                       .withColumnRenamed('product_moisture', 'hum_malt_max')

# Max lower_limit
df_bqt_min = df_bqt.groupBy("fabrication_order_number").pivot("code").agg(F.max("lower_limit"))

# Renommer les colonnes dans df_bqt_min
df_bqt_min = df_bqt_min.withColumnRenamed('malt_color_ebc', 'coloration_EBC_min') \
                       .withColumnRenamed('malt_free_amino_nitrogen', 'FAN_min') \
                       .withColumnRenamed('malt_friability', 'friabilite_min') \
                       .withColumnRenamed('malt_soluble_betaglucan', 'betaG_min') \
                       .withColumnRenamed('product_moisture', 'hum_malt_min')

# Jointure
df_bqt = df_bqt_max.join(df_bqt_min.select('fabrication_order_number', 'coloration_EBC_min', 'FAN_min', 'friabilite_min', 'betaG_min', 'hum_malt_min'), on='fabrication_order_number', how='left')

# Sélection des colonnes finales
df_bqt = df_bqt.select(
    'fabrication_order_number',
    'FAN_max',
    'FAN_min',
    'hum_malt_max',
    'hum_malt_min',
    'betaG_max',
    'betaG_min',
    'friabilite_max',
    'friabilite_min',
    'coloration_EBC_max',
    'coloration_EBC_min'
)

# Créer une vue temporaire pour le DataFrame
df_bqt.createOrReplaceTempView("batches_quality_targets_min_max")

indice_qualite = f"""
SELECT
    b.id_batch,
    b.batch_number AS batch,
    malt_free_amino_nitrogen AS FAN,
    malt_soluble_beta_glucan AS betaG,
    malt_friability AS friabilite,
    malt_moisture AS hum_malt,
    malt_color_ebc AS coloration_EBC,
    bqtmm.*
FROM {source}.batches b
JOIN {source_gold}.malt_quality_results mqr ON mqr.fabrication_order = b.fabrication_order_number
JOIN batches_quality_targets_min_max bqtmm ON bqtmm.fabrication_order_number = mqr.fabrication_order
"""

df_indice_qualite = spark.sql(indice_qualite)

# Si dans les bornes alors = 0,2 sinon 0 puis somme finale des 5 critères 
df_indice_qualite = df_indice_qualite.withColumn('FAN_result', F.when((F.col('FAN_min') <= F.col('FAN')) & (F.col('FAN') <= F.col('FAN_max')), 20).otherwise(0))
df_indice_qualite = df_indice_qualite.withColumn('hum_malt_result', F.when((F.col('hum_malt_min') <= F.col('hum_malt')) & (F.col('hum_malt') <= F.col('hum_malt_max')), 20).otherwise(0))
df_indice_qualite = df_indice_qualite.withColumn('betaG_result', F.when((F.col('betaG_min') <= F.col('betaG')) & (F.col('betaG') <= F.col('betaG_max')), 20).otherwise(0))
df_indice_qualite = df_indice_qualite.withColumn('friabilite_result', F.when((F.col('friabilite_min') <= F.col('friabilite')) & (F.col('friabilite') <= F.col('friabilite_max')), 20).otherwise(0))
df_indice_qualite = df_indice_qualite.withColumn('coloration_EBC_result', F.when((F.col('coloration_EBC_min') <= F.col('coloration_EBC')) & (F.col('coloration_EBC') <= F.col('coloration_EBC_max')), 20).otherwise(0))
df_indice_qualite = df_indice_qualite.withColumn('Indice qualite', F.round(F.col('FAN_result') + F.col('hum_malt_result') + F.col('betaG_result') + F.col('friabilite_result') + F.col('coloration_EBC_result'),1))

df_indice_qualite = df_indice_qualite.select('id_batch', 'batch', 'Indice qualite')

# Dépivoter la colonne 'Indice qualite'
df_indice_qualite = df_indice_qualite.selectExpr("id_batch", "batch", "'Indice qualite' as parameter", "`Indice qualite` as value").dropna(subset=["value"])

# Conversion de toutes les colonnes en STRING
df_indice_qualite = df_indice_qualite.select([col(c).cast("STRING").alias(c) for c in df_quality.columns])

# Ajoute % derrière chaque valeur
df_indice_qualite = df_indice_qualite.withColumn("value",concat(col("value").cast("string"), lit("%")))

In [0]:
# Concatenation des DataFrames
df_batch_report = df_main.unionByName(df_parameter_goods_malt) \
                   .unionByName(df_r2_sec) \
                   .unionByName(df_parameter_categ_1) \
                   .unionByName(df_manual_entries) \
                   .unionByName(df_quality) \
                   .unionByName(df_indice_qualite)

# Supprime les lignes vident dans la colonne 'value'
df_batch_report = df_batch_report.filter(df_batch_report["value"].isNotNull())

# Remplace les '.' par ',' dans la colonne 'value'
df_batch_report = df_batch_report.withColumn("value", regexp_replace(col("value"), r"\.", ","))

# Supprime les doublons
df_batch_report = df_batch_report.dropDuplicates(['id_batch', 'parameter'])


# Supprime les colonnes suivantes --> modif suite demande William réorganisation des colonnes
to_drop = [
    "Lot mes",
    "Site de production",
    "Moisture Malt",
    "Indice qualite",
    "#_volume",
    "Date de calibrage",
    "Row Material Storage Duration",
    ]

df_batch_report = df_batch_report.filter(col("parameter").isNull() | (~trim(col("parameter")).isin(to_drop)))


# Colonne 'custom_order'
df_batch_report = df_batch_report.withColumn(
    "custom_order",
    when(trim(col("parameter")) == "Ordre de fabrication", 1)
    .when(trim(col("parameter")) == "Type de production", 2)
    .when(trim(col("parameter")) == "Année de récolte", 3)
    .when(trim(col("parameter")) == "Début de production", 4)
    .when(trim(col("parameter")) == "Fin de production", 5)
    .when(trim(col("parameter")) == "Cahier des charges", 6)
    .when(trim(col("parameter")) == "Espèce", 7)
    .when(trim(col("parameter")) == "Variété", 8)
    .when(trim(col("parameter")) == "Malt Quantity", 9)
    .when(trim(col("parameter")) == "R2 Yield", 10)
    .when(trim(col("parameter")) == "R2 Dry", 11)
    .when(trim(col("parameter")) == "Moisture Malt Quality", 12)
    .when(trim(col("parameter")) == "Friabilité", 13)
    .when(trim(col("parameter")) == "Couleur EBC", 14)
    .when(trim(col("parameter")) == "FAN", 15)
    .when(trim(col("parameter")) == "Betaglucanes", 16)
    .when(trim(col("parameter")) == "Viscosité", 17)
    .when(trim(col("parameter")) == "Row Material quantity", 18)
    .when(trim(col("parameter")) == "Moisture-RCP", 19)
    .when(trim(col("parameter")) == "Barley Protein Content", 20)
    .when(trim(col("parameter")) == "GL 4mL 72h", 21)
    .when(trim(col("parameter")) == "GL 8mL 72h", 22)
    .when(trim(col("parameter")) == "Broken Kernels", 23)
    .when(trim(col("parameter")) == "CAL < 2,2 MM-RM", 24)
    .when(trim(col("parameter")) == "#_mass", 25)
    .when(trim(col("parameter")) == "#_doses", 26)
    .when(trim(col("parameter")) == "Moisture Tr1", 27)
    .when(trim(col("parameter")) == "Moisture Tr2", 28)
    .when(trim(col("parameter")) == "Humidité prégermination", 29)
    .when(trim(col("parameter")) == "Trempe 2 - grains piqués", 30)
    .when(trim(col("parameter")) == "Moisture Malt Vert J1", 31)
    .when(trim(col("parameter")) == "Germ 1 - grains piqués", 32)
    .when(trim(col("parameter")) == "Germ 2 - grains piqués manquants", 33)
    .when(trim(col("parameter")) == "Moisture Malt Vert J2", 34)
    .when(trim(col("parameter")) == "Humidité malt vert J3", 35)
    .when(trim(col("parameter")) == "Humidité malt vert J4", 36)
    .when(trim(col("parameter")) == "Moisture Before Kilning", 37)
    .otherwise(99)
)

In [0]:
# Ajouter les colonnes 'created_at', 'updated_at' et 'deleted_at' dans le dataframe
df_batch_report = df_batch_report \
    .withColumn('created_at', lit(None).cast('timestamp')) \
    .withColumn('updated_at', lit(None).cast('timestamp')) \
    .withColumn('deleted_at', lit(None).cast('timestamp'))

In [0]:
if verbose_mode == 'debug':
    print( list(df_batch_report.columns))
    print(df_batch_report.dtypes)

In [0]:
target_table_df = spark.sql(f"""SELECT * FROM {current_catalog}.{current_schema}.{current_process} WHERE 1=0""").toPandas()

if verbose_mode == 'debug':
    display(target_table_df.columns)
    print( list(target_table_df.columns))
    print(target_table_df.dtypes)

In [0]:
if not list(sorted(df_batch_report.columns)) == list(sorted(target_table_df.columns)):
    print("Erreur : Le fichier n'a pas les mêmes colonnes que la table.")
    dbutils.notebook.exit(-1)
else:
    print("Check columns : OK")

In [0]:
df_batch_report.createOrReplaceTempView(f"v_ingestion_{current_process}")
if verbose_mode == 'debug' :
    display(df_batch_report)

In [0]:
business_keys = [('id_batch',), ('parameter',)]

In [0]:
excluded_fields = [('created_at',), ('updated_at',), ('deleted_at',)]

In [0]:
data_found = delta_table_sql_update_table(f"v_ingestion_{current_process}", f"{current_catalog}.{current_schema}.{current_process}", business_keys, excluded_fields, "upsert", verbose_mode)
if verbose_mode == 'debug':
    display(f"Merge Result : {data_found}") 
if data_found==-1:
    print("erreur de merge")
    dbutils.notebook.exit()

dim_batch

In [0]:
current_process="dim_batch"

In [0]:
dim_batch = f"""
SELECT
    b.id_batch,
    b.batch_number,
    b.mes_number,
    b.fabrication_order_number AS ordre_fabrication,
    b.harvest AS annee_recolte,
    b.production_line AS id_site,
    b.requirement_specifications AS id_cdc,
    b.production_type AS id_type,
    b.variety AS id_variete,
    b.planned_date AS debut_de_production,
    dfp.fin_de_production
FROM {source}.batches b
LEFT JOIN date_fin_production dfp ON b.id_batch = dfp.id_batch
WHERE b.deleted = false
"""

df_dim_batch = spark.sql(dim_batch)

# Modification de la colonne 'batch' en majuscule
df_dim_batch = df_dim_batch.withColumn("batch_number", upper(df_dim_batch["batch_number"]))

In [0]:
# Ajouter les colonnes 'created_at', 'updated_at' et 'deleted_at' dans le dataframe
df_dim_batch = df_dim_batch \
    .withColumn('created_at', lit(None).cast('timestamp')) \
    .withColumn('updated_at', lit(None).cast('timestamp')) \
    .withColumn('deleted_at', lit(None).cast('timestamp'))

In [0]:
if verbose_mode == 'debug':
    print( list(df_dim_batch.columns))
    print(df_dim_batch.dtypes)

In [0]:
target_table_df = spark.sql(f"""SELECT * FROM {current_catalog}.{current_schema}.{current_process} WHERE 1=0""").toPandas()

if verbose_mode == 'debug':
    display(target_table_df.columns)
    print( list(target_table_df.columns))
    print(target_table_df.dtypes)

In [0]:
if not list(sorted(df_dim_batch.columns)) == list(sorted(target_table_df.columns)):
    print("Erreur : Le fichier n'a pas les mêmes colonnes que la table.")
    dbutils.notebook.exit(-1)
else:
    print("Check columns : OK")

In [0]:
df_dim_batch.createOrReplaceTempView(f"v_ingestion_{current_process}")
if verbose_mode == 'debug' :
    display(df_dim_batch)

In [0]:
business_keys = [('id_batch',)]

In [0]:
excluded_fields = [('created_at',), ('updated_at',), ('deleted_at',)]

In [0]:
data_found = delta_table_sql_update_table(f"v_ingestion_{current_process}", f"{current_catalog}.{current_schema}.{current_process}", business_keys, excluded_fields, "upsert", verbose_mode)
print(data_found)
if verbose_mode == 'debug':
    display(f"Merge Result : {data_found}") 
if data_found==-1:
    print("erreur de merge")
    dbutils.notebook.exit(-1)